<a id="abstract"></a>
# Abstract

Machine Learning models deployed in production frequently lose performance over time due to concept drift: the change in the statistical relationship between input variables and the target variable. In meteorological time series, this phenomenon is particularly relevant, as seasonality, climate regime changes, and observational noise continuously alter the data distribution. This study investigates how different model update policies handle this non-stationarity in a real problem: binary hourly rainfall prediction at Miami International Airport, using 14 years of observations from the ASOS network (2012–2025).

Three strategies were compared under identical experimental conditions. Same time windows, same data, and same evaluation process: (i) **incremental learning**, with continuous model updates via online learning; (ii) **periodic retraining**, with model reconstruction at each new window; and (iii) **single training**, a static baseline. The evaluation followed a sliding window scheme (180 days of training, 14 days of testing, 14-day step), preserving the temporal sequence. Given the imbalance of the positive class, the main metric adopted was the F1-score, complemented by precision, recall, and accuracy.

The results show that the **Incremental** scenario obtained the highest aggregated F1 (0.470), outperforming Retraining (0.381) and Single Training (0.333), with greater regularity over the 14 years evaluated. The paired Wilcoxon test, with Bonferroni correction, confirmed statistically significant differences with large effect size between the Incremental scenario and the others. It is concluded that, in real temporal problems such as rainfall prediction, the model update policy is a central part of the solution, and not a secondary engineering detail.

References

J. Gama, I. Žliobaitė, A. Bifet, M. Pechenizkiy, and A. Bouchachia. A Survey on Concept Drift Adaptation. ACM Computing Surveys, 46(4), Article 44, 2014. DOI: 10.1145/2523813.

J. Lu, A. Liu, F. Dong, F. Gu, J. Gama, and G. Zhang. Learning under Concept Drift: A Review. IEEE Transactions on Knowledge and Data Engineering, 31(12), 2346–2363, 2019. DOI: 10.1109/TKDE.2018.2876857.

Iowa Environmental Mesonet (IEM), Iowa State University. ASOS / METAR data download interface. Available at: https://mesonet.agron.iastate.edu/request/download.phtml.


# Table of Contents

- [1. Introduction](#sec-introduction)
  - [Objectives](#sec-objectives)
  - [1.2 Problem Statement and Hypothesis](#sec-problem-hypothesis)
  - [1.2.1 Why F1-Score Was Chosen as the Primary Metric](#sec-f1)
- [2. Methodology](#sec-methodology)
  - [2.1 Compared Scenarios](#sec-scenarios)
  - [2.2 Experimental Setup and Methodological Choices](#sec-environment)
  - [2.3 Dataset and Data Preprocessing](#sec-dataset)
    - [2.3.1 Data Preparation](#sec-data-preparation)
    - [2.3.2 Temporal Validation Strategy](#sec-temporal-validation)
  - [2.4 Evaluation Metrics](#sec-evaluation-metrics)
  - [2.5 Results Structure](#sec-results-structure)
  - [2.6 Scenario 1 — Incremental Learning](#sec-incremental)
  - [2.7 Scenario 2 — Periodic Retraining](#sec-periodic-retraining)
  - [2.8 Scenario 3 — Static Training](#sec-static-training)
- [3. Experiment Execution](#sec-execution)
- [4. Results](#sec-results)
  - [4.1 Temporal Dynamics of the Target Variable](#sec-target-dynamics)
  - [4.2 Results Consolidation](#sec-results-consolidation)
  - [4.3 Interpreting the Aggregated Results](#sec-aggregated-results)
  - [4.4 Statistical Comparison Between Scenarios](#sec-statistical-comparison)
  - [4.5 Statistical Interpretation](#sec-statistical-interpretation)
- [5. Discussion](#sec-discussion)
  - [5.1 Experiment Limitations](#sec-limitations)
  - [5.2 Main Results](#sec-main-results)
- [6. Conclusion](#sec-conclusion)

<a id="sec-introduction"></a>
# 1. Introduction

Machine Learning models deployed in production face a recurring challenge: the world changes, but the trained model remains the same. This phenomenon, known as **concept drift**, occurs when the statistical relationship between the input variables and the target changes over time, whether due to seasonality, shifts in weather patterns, sensor noise, or any other non-stationary factor.

This issue is particularly relevant in meteorological time series: a model trained on patterns from one period may experience performance degradation months later simply because the underlying data distribution has evolved. The practical question that motivates this study is: **which model update strategy best handles these changes over time?**

This notebook investigates this question using a real-world binary rainfall prediction problem based on hourly observations from Miami International Airport, comparing three model update strategies under exactly the same experimental conditions.

<a id="sec-objectives"></a>
## Objectives

- Compare three model update strategies (static model, periodic retraining, and incremental learning) in a real-world time series classification problem.
- Evaluate not only the overall predictive performance but also the **performance stability over time** of each strategy.
- Validate the observed differences using a paired statistical test.

In [1]:
# Optional dependency: uncomment the line below if the "river" package
# is not yet installed in the execution environment.
# !{sys.executable} -m pip install river -q

In [34]:
# Standard libraries
import sys
import time
from dataclasses import dataclass
from typing import Dict, List, Tuple
from IPython.display import display

In [35]:
# External libraries
import numpy as np
import pandas as pd
from river import compose, linear_model, optim, preprocessing
from scipy.stats import wilcoxon
from sklearn.metrics import confusion_matrix
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
import plotly.graph_objects as go

In [29]:
# Visualization settings

# Consistent color palette by scenario, reused across all figures.
SCENARIO_COLORS: Dict[str, str] = {
    "Incremental Learning": "#2ca02c",        # green
    "Periodic Retraining": "#1f77b4", # blue
    "Static Training": "#d62728",     # red
}


def apply_layout(
    fig: go.Figure,
    title: str,
    height: int = 450,
    width: int = 950,
) -> go.Figure:
    """Applies a standardized visual layout to Plotly figures in the notebook.

    Args:
        fig: Plotly figure (Express or Graph Objects) to be formatted.
        title: Title displayed at the top of the figure.
        height: Figure height in pixels.
        width: Figure width in pixels.

    Returns:
        go.Figure: The same figure received with the updated layout applied.

    Notes:
        Centralizes styling choices (font, grid, horizontal legend) to maintain
        visual consistency across all figures in the study.
    """
    fig.update_layout(
        title=dict(text=title, x=0.5, xanchor="center", font=dict(size=16)),
        template="plotly_white",
        font=dict(family="Segoe UI, Helvetica, Arial, sans-serif", size=13),
        height=height,
        width=width,
        margin=dict(l=60, r=30, t=70, b=50),
        legend=dict(
            orientation="h",
            yanchor="bottom",
            y=1.02,
            xanchor="center",
            x=0.5,
        ),
        hovermode="x unified",
    )
    fig.update_xaxes(showgrid=True, gridcolor="rgba(0,0,0,0.08)")
    fig.update_yaxes(showgrid=True, gridcolor="rgba(0,0,0,0.08)")
    return fig


# pio.renderers.default = "notebook"
pio.renderers.default = "iframe"


def render_fig(fig: go.Figure) -> None:
    """Displays a Plotly figure.

    Args:
        fig: Plotly figure to be displayed.

    Returns:
        None.
    """
    try:
        fig.show(renderer=pio.renderers.default)
    except Exception:
        display(fig)

<a id="sec-problem-hypothesis"></a>
## 1.2 Problem Statement and Hypothesis

Rainfall prediction is inherently a challenging task. Even in a binary classification setting, the relationship between meteorological variables and the occurrence of rainfall may change over time due to seasonal effects, shifts in weather patterns, and observational noise.

The central hypothesis of this study is straightforward: **in a long-term temporal prediction problem, model update strategies are expected to adapt better than a static approach**.

Accordingly, the research question is defined as:

> Which model update strategy maintains the best predictive performance over time when applying incremental logistic regression to real-world precipitation data?

<a id="sec-f1"></a>
### 1.2.1 Why F1-Score Was Chosen as the Primary Metric

Rainfall events represent the minority class in this dataset, as most hourly observations correspond to periods without precipitation. In this context, **accuracy can be misleading**. A model that always predicts "no rain" would correctly classify most observations while failing to identify any actual rainfall events.

For this reason, the primary evaluation metric adopted in this study is the **F1-score**, which balances:

- **Precision**: of the observations predicted as rain, how many actually corresponded to rainfall;
- **Recall**: of the actual rainfall events, how many were correctly identified by the model.

The F1-score penalizes models that achieve high precision at the expense of recall, or vice versa, making it a more informative metric than accuracy alone for imbalanced classification problems such as rainfall prediction.

In [6]:
# Experiment constants: column names, decision threshold, and
# sliding temporal window configuration.
TARGET_RAW: str = "p01i"
TARGET_BINARY: str = "rain_event"
FEATURES: List[str] = ["tmpf", "dwpf", "relh", "drct", "sknt"]

THRESHOLD: float = 0.30
TRAIN_DAYS: int = 180
TEST_DAYS: int = 14
STEP_DAYS: int = 14
REGION: str = "MIA"

<a id="sec-methodology"></a>
# 2. Methodology

<a id="sec-scenarios"></a>
## 2.1 Compared Scenarios

The three scenarios evaluated in this study use exactly the same sequence of temporal windows. This ensures methodological consistency and prevents performance differences from being attributed to different train–test splits.

| Scenario | Description | Represents |
|---|---|---|
| Incremental Learning | The model is initially trained and then continuously updated as new observations become available. | Continuous learning with accumulated knowledge. |
| Periodic Retraining | The model is rebuilt from scratch at each new training window. | Periodic model updates without retaining knowledge from previous windows. |
| Static Training | The model is trained only once and reused throughout the entire evaluation period without further updates. | A static baseline used to assess the practical value of temporal adaptation. |

Before describing each strategy in detail, it is important to emphasize a key methodological aspect of this study: **all three scenarios share exactly the same temporal windows, the same dataset, and the same evaluation procedure**. The only element that differs among them is the model update policy. This standardization ensures a fair comparison, allowing any observed differences in performance to be attributed to the update strategy itself rather than to differences in data partitioning.

In [7]:
@dataclass
class WindowConfig:
    """Configuration of the sliding temporal windows used for evaluation.

    Attributes:
        train_days: Number of days used for training in each window.
        test_days: Number of days used for testing in each window.
        step_days: Number of days between the start of consecutive windows.
    """
    train_days: int = TRAIN_DAYS
    test_days: int = TEST_DAYS
    step_days: int = STEP_DAYS


def make_model() -> compose.Pipeline:
    """Creates a new instance of the incremental logistic regression pipeline.

    Returns:
        compose.Pipeline: River pipeline containing feature standardization
        followed by logistic regression trained with SGD.

    Notes:
        The same hyperparameter configuration is used across all scenarios
        to ensure comparability between them.
    """
    return compose.Pipeline(
        preprocessing.StandardScaler(),
        linear_model.LogisticRegression(
            optimizer=optim.SGD(lr=0.005),
            loss=optim.losses.Log(),
            l2=1e-4,
            intercept_lr=0.005,
            clip_gradient=1e12,
        ),
    )

<a id="sec-environment"></a>
## 2.2 Experimental Setup and Methodological Choices

The notebook was structured to ensure that the experimental workflow is both reproducible and easy to interpret. Data preparation, temporal window definition, model training, evaluation, and statistical analysis were organized into separate sections.

The main methodological decisions are summarized below:

- the task was formulated as a binary rainfall classification problem;
- model evaluation was performed using **temporal partitioning** rather than random train–test splits;
- the original class distribution was preserved, with no artificial class balancing applied;
- the final analysis focuses on **F1-score, precision, and recall**, which are more informative metrics for imbalanced classification problems.

<a id="sec-dataset"></a>
## 2.3 Dataset and Data Preprocessing

<a id="sec-data-preparation"></a>
### 2.3.1 Data Preparation

The dataset was filtered to include only observations from the selected weather station at Miami International Airport and transformed into a clean analytical dataset, chronologically ordered and ready for model development.

During this stage, missing values and special codes were handled, predictor variables were converted to numeric format, and a binary target variable indicating rainfall occurrence was created. The preprocessing pipeline was intentionally kept simple and transparent, avoiding unnecessary transformations that could reduce the interpretability of the model.

In [8]:
def prepare_region_data(asos_raw: pd.DataFrame) -> pd.DataFrame:
    """Filters, cleans, and enriches raw data from the selected weather station.

    Args:
        asos_raw: Raw DataFrame containing meteorological observations from
            all stations in the ASOS network.

    Returns:
        pd.DataFrame: Analytical dataset filtered for the station defined
        in REGION, chronologically ordered, containing the binary rainfall
        target variable and precomputed date-related auxiliary columns.

    Raises:
        ValueError: If the resulting DataFrame is empty after data cleaning.

    Notes:
        Special missing-value codes ("M") and trace rainfall values ("T") are
        handled before numeric conversion, preserving the original semantics
        of the meteorological observations.
    """
    df = asos_raw.copy()
    df = df[df["station"] == REGION].copy()

    df[TARGET_RAW] = (
        df[TARGET_RAW]
        .astype(str)
        .str.strip()
        .replace({"M": np.nan, "": np.nan, "T": "0.00"})
    )
    df[TARGET_RAW] = pd.to_numeric(df[TARGET_RAW], errors="coerce")

    for col in FEATURES:
        df[col] = df[col].astype(str).str.strip().replace({"M": np.nan, "": np.nan})
        if col == "drct":
            df[col] = df[col].replace({"999": np.nan, "VRB": np.nan})
        df[col] = pd.to_numeric(df[col], errors="coerce")

    df["valid"] = pd.to_datetime(df["valid"], utc=True, errors="coerce")
    df = df.dropna(subset=FEATURES + [TARGET_RAW, "valid"]).copy()
    df = df.sort_values("valid").reset_index(drop=True)

    if df.empty:
        raise ValueError(
            f"No valid records found for station '{REGION}' "
            "after data cleaning."
        )

    df[TARGET_BINARY] = df[TARGET_RAW] >= 0.01
    df["year"] = df["valid"].dt.year
    df["month"] = df["valid"].dt.month
    df["year_month"] = df["valid"].dt.tz_convert(None).dt.to_period("M").astype(str)

    min_valid = df["valid"].min()
    df["month_seq"] = 12 * (df["year"] - min_valid.year) + (df["month"] - min_valid.month) + 1
    df["day_seq"] = (df["valid"] - min_valid).dt.days + 1

    keep_cols = [
        "station", "valid", TARGET_RAW, TARGET_BINARY, *FEATURES,
        "year", "month", "year_month", "month_seq", "day_seq"
    ]
    return df[keep_cols].copy()

<a id="sec-temporal-validation"></a>
### 2.3.2 Temporal Validation Strategy

Because the prediction task has a temporal nature, model evaluation does not rely on random train–test splits. Instead, the experiment adopts a **sliding window** approach, where the model is always trained on historical data and evaluated on the immediately following time period.

Each iteration uses:

- **180 days** for training;
- **14 days** for testing;
- a **14-day step** between consecutive windows.

This configuration preserves the chronological order of the observations and provides a more realistic approximation of how the model would be deployed in a real-world forecasting scenario.

In [9]:
def create_flexible_windows(
    df: pd.DataFrame,
    config: WindowConfig,
) -> Tuple[pd.DataFrame, List[Dict]]:
    """Generates sliding train/test windows over the time series.

    Args:
        df: Chronologically ordered DataFrame containing the auxiliary
            "day_seq" column generated by `prepare_region_data`.
        config: Configuration defining training size, testing size, and
            step between consecutive windows.

    Returns:
        Tuple[pd.DataFrame, List[Dict]]: The reordered DataFrame and a list
        of generated windows, each containing training and testing indices
        along with their respective start and end dates.

    Notes:
        The window always moves forward in time (training on past data and
        testing on the immediate future), preserving the temporal sequence
        required for time series experiments.
    """
    df = df.sort_values("valid").reset_index(drop=True).copy()
    windows = []

    current_seq = int(df["day_seq"].min())
    max_seq = int(df["day_seq"].max())

    while current_seq + config.train_days + config.test_days - 1 <= max_seq:
        train_start_seq = current_seq
        train_end_seq = current_seq + config.train_days - 1
        test_start_seq = train_end_seq + 1
        test_end_seq = test_start_seq + config.test_days - 1

        train_mask = (df["day_seq"] >= train_start_seq) & (df["day_seq"] <= train_end_seq)
        test_mask = (df["day_seq"] >= test_start_seq) & (df["day_seq"] <= test_end_seq)

        windows.append({
            "train_idx": df.index[train_mask].tolist(),
            "test_idx": df.index[test_mask].tolist(),
            "train_start_date": df.loc[train_mask, "valid"].iloc[0],
            "train_end_date": df.loc[train_mask, "valid"].iloc[-1],
            "test_start_date": df.loc[test_mask, "valid"].iloc[0],
            "test_end_date": df.loc[test_mask, "valid"].iloc[-1],
        })

        current_seq += config.step_days

    return df, windows


def validate_windows(df: pd.DataFrame, windows: List[Dict]) -> bool:
    """Validates the temporal integrity of the generated windows.

    Args:
        df: Reference DataFrame used to generate the windows.
        windows: List of windows produced by
            `create_flexible_windows`.

    Returns:
        bool: True if all generated windows are valid.

    Raises:
        AssertionError: If any window contains overlap between training and
            testing indices, or if the training period does not fully precede
            the testing period.

    Notes:
        This validation prevents data leakage between training and testing
        sets, ensuring that the temporal order is preserved.
    """
    for w in windows:
        tr = df.loc[w["train_idx"]]
        te = df.loc[w["test_idx"]]
        assert set(w["train_idx"]).isdisjoint(set(w["test_idx"]))
        assert tr["valid"].max() < te["valid"].min()
    return f"{len(windows)} windows successfully validated. Training and testing periods are temporally consistent."

<a id="sec-evaluation-metrics"></a>
## 2.4 Evaluation Metrics

Rather than storing only aggregated performance metrics, the notebook records the components of the confusion matrix for every evaluation window. This makes it possible to recompute precision, recall, and F1-score at different levels of aggregation, such as per window, per year, or across the entire experiment.

This design choice is particularly important because the prediction task is characterized by class imbalance. In this setting, **accuracy alone can be misleading**, whereas F1-score, precision, and recall provide a more informative assessment of the model's ability to detect the positive class.

In [10]:
def predict_with_threshold(
    model,
    X_df: pd.DataFrame,
    threshold: float = 0.5,
) -> Tuple[np.ndarray, np.ndarray]:
    """Generates binary predictions from model probabilities.

    Args:
        model: Incremental model compatible with the River API
            (must expose `predict_proba_one`).
        X_df: DataFrame containing input features, one row per
            observation.
        threshold: Decision threshold applied to the probability of the
            positive class.

    Returns:
        Tuple[np.ndarray, np.ndarray]: Boolean prediction array and an
        array containing the associated probabilities for the positive
        class.

    Notes:
        Predictions are generated one sample at a time, respecting the
        online nature of the incremental model.
    """
    probs, preds = [], []

    for x in X_df.to_dict("records"):
        proba_dict = model.predict_proba_one(x)
        p1 = proba_dict.get(True, 0.0)
        probs.append(p1)
        preds.append(p1 >= threshold)

    return np.array(preds, dtype=bool), np.array(probs)


def confusion_summary(y_true: np.ndarray, y_pred: np.ndarray) -> Dict:
    """Computes the components of the binary confusion matrix.

    Args:
        y_true: Boolean array containing the actual labels.
        y_pred: Boolean array containing the predicted labels.

    Returns:
        Dict: Dictionary containing the counts of true negatives (tn),
        false positives (fp), false negatives (fn), and true positives (tp).
    """
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[False, True]).ravel()
    return {"tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp)}


def compute_metrics_from_confusion(tp, tn, fp, fn) -> pd.DataFrame:
    """Derives accuracy, precision, recall, and F1-score from the confusion matrix.

    Args:
        tp: Number(s) of true positives.
        tn: Number(s) of true negatives.
        fp: Number(s) of false positives.
        fn: Number(s) of false negatives.

    Returns:
        pd.DataFrame: DataFrame containing the columns "accuracy",
        "precision", "recall", and "f1", calculated in a vectorized manner.

    Notes:
        Division-by-zero cases are handled using `np.where`, returning 0.0
        for windows without sufficient support instead of raising errors.
    """
    total = tp + tn + fp + fn
    accuracy = np.where(total > 0, (tp + tn) / total, 0.0)
    precision = np.where(tp + fp > 0, tp / (tp + fp), 0.0)
    recall = np.where(tp + fn > 0, tp / (tp + fn), 0.0)
    f1 = np.where(
        precision + recall > 0,
        2 * precision * recall / (precision + recall),
        0.0,
    )

    return pd.DataFrame({
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1,
    })


def evaluate_model_on_window(
    model,
    test_data: pd.DataFrame,
    threshold: float = 0.5,
) -> Dict:
    """Evaluates a model on a test window and measures inference time.

    Args:
        model: Previously trained incremental model compatible with River.
        test_data: DataFrame containing the test window, including the
            input features and binary target variable.
        threshold: Decision threshold used to convert probabilities into
            binary predictions.

    Returns:
        Dict: Confusion matrix metrics, positive class support, number of
        samples, inference time, and arrays containing true labels,
        predictions, and probabilities.
    """
    
    y_true = test_data[TARGET_BINARY].astype(bool).to_numpy()
    y_pred, y_prob = predict_with_threshold(model, test_data[FEATURES], threshold)

    metrics = confusion_summary(y_true, y_pred)

    class1_count = int(np.sum(y_true))
    metrics.update({
        "class1_support": class1_count,
        "class1_in_test": class1_count > 0,
        "n_test_samples": len(y_true),
        "y_true": y_true,
        "y_pred": y_pred,
        "y_prob": y_prob,
    })
    return metrics

<a id="sec-results-structure"></a>
## 2.5 Results Structure

Each evaluation window produces a standardized record containing the corresponding time interval, the confusion matrix components, the support of the positive class, and the prediction artifacts. This standardized structure facilitates experiment auditing and provides greater flexibility for subsequent analyses and performance comparisons.

In [11]:
def build_result_row(
    window_idx: int,
    window_info: Dict,
    metrics_info: Dict,
    scenario: str,
) -> Dict:
    """Builds a standardized result row for an evaluated window.

    Args:
        window_idx: Sequential index of the window within the scenario.
        window_info: Dictionary containing the training and testing dates
            of the window, generated by `create_flexible_windows`.
        metrics_info: Dictionary of metrics returned by
            `evaluate_model_on_window`.
        scenario: Name of the evaluated scenario (e.g., "Incremental").

    Returns:
        Dict: Consolidated record ready to compose the experiment results
        DataFrame.
    """
    return {
        "window_idx": window_idx,
        "scenario": scenario,
        "train_start": str(window_info["train_start_date"]),
        "train_end": str(window_info["train_end_date"]),
        "test_start": str(window_info["test_start_date"]),
        "test_end": str(window_info["test_end_date"]),
        "tp": metrics_info["tp"],
        "tn": metrics_info["tn"],
        "fp": metrics_info["fp"],
        "fn": metrics_info["fn"],
        "class1_support": metrics_info["class1_support"],
        "class1_in_test": metrics_info["class1_in_test"],
        "n_test_samples": metrics_info["n_test_samples"],
        "y_true": metrics_info["y_true"],
        "y_pred": metrics_info["y_pred"],
        "y_prob": metrics_info["y_prob"],
    }

<a id="sec-incremental"></a>
## 2.6 Scenario 1 — Incremental Learning

In the first scenario, the model is trained using the initial training window and is subsequently updated with the most recent observations before generating predictions for each new evaluation window.

This scenario represents a continuous learning workflow, in which the model retains the knowledge acquired throughout the experiment and gradually adapts to changes in the underlying data distribution over time.

In [22]:
def run_scenario_incremental(
    df: pd.DataFrame,
    windows: List[Dict],
    threshold: float = 0.5,
) -> pd.DataFrame:
    """Runs the incremental learning scenario across all evaluation windows.

    The model is trained once using the first training window and is then
    continuously updated with the test data from each previous window before
    predicting the following window.

    Args:
        df: Prepared DataFrame containing the feature columns and the
            binary target variable.
        windows: List of temporal windows generated by
            `create_flexible_windows`.
        threshold: Decision threshold used to convert probabilities into
            binary predictions.

    Returns:
        pd.DataFrame: One record per evaluated window, following the format
        generated by `build_result_row`.

    Notes:
        This function uses online learning (`learn_one`) and maintains the
        accumulated model state across consecutive windows.
    """
    rows = []
    model = make_model()
    scenario = "Incremental Learning"

    w0 = windows[0]
    train0 = df.loc[w0["train_idx"]].copy()
    test0 = df.loc[w0["test_idx"]].copy()

    for x, y in zip(
        train0[FEATURES].to_dict("records"),
        train0[TARGET_BINARY].astype(bool),
    ):
        model.learn_one(x, y)

    metrics_info = evaluate_model_on_window(model, test0, threshold)
    rows.append(build_result_row(1, w0, metrics_info, scenario))

    previous_test = test0.copy()

    for i, w in enumerate(windows[1:], start=2):
        for x, y in zip(
            previous_test[FEATURES].to_dict("records"),
            previous_test[TARGET_BINARY].astype(bool),
        ):
            model.learn_one(x, y)

        current_test = df.loc[w["test_idx"]].copy()
        metrics_info = evaluate_model_on_window(model, current_test, threshold)
        rows.append(build_result_row(i, w, metrics_info, scenario))

        previous_test = current_test.copy()

    return pd.DataFrame(rows)

<a id="sec-periodic-retraining"></a>
## 2.7 Scenario 2 — Periodic Retraining

In the second scenario, the model is rebuilt from scratch for each temporal window, using only the observations available in the corresponding training set.

This strategy represents a more conservative adaptation approach: although it incorporates recent data at every iteration, it does not retain any direct knowledge from previous training windows.

In [13]:
def run_scenario_retrain(
    df: pd.DataFrame,
    windows: List[Dict],
    threshold: float = 0.5,
) -> pd.DataFrame:
    """Runs the periodic retraining scenario across all evaluation windows.

    For each window, a new model is created from scratch and trained only
    with the training data available in that specific window, without
    inheriting knowledge from previous windows.

    Args:
        df: Prepared DataFrame containing the feature columns and the
            binary target variable.
        windows: List of temporal windows generated by
            `create_flexible_windows`.
        threshold: Decision threshold used to convert probabilities into
            binary predictions.

    Returns:
        pd.DataFrame: One record per evaluated window, following the format
        generated by `build_result_row`.

    Notes:
        This strategy represents adaptation without memory between windows:
        each model "forgets" previous knowledge when it is recreated.
    """
    rows = []
    scenario = "Periodic Retraining"

    for i, w in enumerate(windows, start=1):
        train_data = df.loc[w["train_idx"]].copy()
        test_data = df.loc[w["test_idx"]].copy()

        model = make_model()

        for x, y in zip(
            train_data[FEATURES].to_dict("records"),
            train_data[TARGET_BINARY].astype(bool),
        ):
            model.learn_one(x, y)

        metrics_info = evaluate_model_on_window(model, test_data, threshold)
        rows.append(build_result_row(i, w, metrics_info, scenario))

    return pd.DataFrame(rows)

<a id="sec-static-training"></a>
## 2.8 Scenario 3 — Static Training

In the third scenario, the model is trained only once using the initial training window and is then reused to generate predictions for all subsequent evaluation windows without any further updates.

This scenario serves as a static baseline, providing a reference against which the practical benefits of temporal model adaptation can be assessed.

In [14]:
def run_scenario_static(
    df: pd.DataFrame,
    windows: List[Dict],
    threshold: float = 0.5,
) -> pd.DataFrame:
    """Runs the static training scenario (static baseline).

    The model is trained only once using the first training window and is
    then reused without any further updates to predict all subsequent
    evaluation windows.

    Args:
        df: Prepared DataFrame containing the feature columns and the
            binary target variable.
        windows: List of temporal windows generated by
            `create_flexible_windows`.
        threshold: Decision threshold used to convert probabilities into
            binary predictions.

    Returns:
        pd.DataFrame: One record per evaluated window, following the format
        generated by `build_result_row`.

    Notes:
        This scenario serves as a static reference to measure the practical
        value of temporal adaptation in the other approaches.
    """
    rows = []
    model = make_model()
    scenario = "Static Training"

    w0 = windows[0]
    train0 = df.loc[w0["train_idx"]].copy()

    for x, y in zip(
        train0[FEATURES].to_dict("records"),
        train0[TARGET_BINARY].astype(bool),
    ):
        model.learn_one(x, y)

    for i, w in enumerate(windows, start=1):
        test_data = df.loc[w["test_idx"]].copy()
        metrics_info = evaluate_model_on_window(model, test_data, threshold)
        rows.append(build_result_row(i, w, metrics_info, scenario))

    return pd.DataFrame(rows)

<a id="sec-execution"></a>
# 3. Experiment Execution

With the methodology and functions in place, the experiment begins by loading the raw dataset, preparing the data, and generating the temporal evaluation windows. The three scenarios are then executed using exactly the same evaluation framework, ensuring that the only difference among them is the model update strategy.

In [23]:
# Loads the raw dataset, prepares the data, and generates the temporal windows.
DATA_PATH = "MIA_2012_2025.csv"

asos_raw = pd.read_csv(DATA_PATH, low_memory=False)
df = prepare_region_data(asos_raw)
df, windows = create_flexible_windows(df, WindowConfig())
print(validate_windows(df, windows))

# Runs the three scenarios using exactly the same window structure.
results_incremental = run_scenario_incremental(df, windows, THRESHOLD)
results_retrain = run_scenario_retrain(df, windows, THRESHOLD)
results_static = run_scenario_static(df, windows, THRESHOLD)

results = pd.concat(
    [results_incremental, results_retrain, results_static],
    ignore_index=True,
)

results["test_start"] = pd.to_datetime(
    results["test_start"],
    utc=True,
    errors="coerce",
)
results["year"] = results["test_start"].dt.year

350 windows successfully validated. Training and testing periods are temporally consistent.


<a id="sec-results"></a>
# 4. Results

<a id="sec-target-dynamics"></a>
## 4.1 Temporal Dynamics of the Target Variable

Before comparing the three model update strategies, it is useful to examine the behavior of the target variable throughout the study period. This analysis provides insight into seasonal patterns, the magnitude of temporal variation, and potential regime changes that may directly influence model performance.

In addition, the dataset exhibits class imbalance, and the analysis was intentionally conducted using the original class distribution without any resampling techniques. This choice makes the experimental results more representative of a real-world deployment scenario.

In [24]:
monthly_precip = (
    df.groupby("year_month", as_index=False)
      .agg(hours=("rain_event", "size"), rainy_hours=("rain_event", "sum"))
)
monthly_precip["year_month_dt"] = pd.to_datetime(monthly_precip["year_month"])
monthly_precip["pct_rain_hours"] = 100 * monthly_precip["rainy_hours"] / monthly_precip["hours"]
monthly_precip["year"] = monthly_precip["year_month_dt"].dt.year
monthly_precip["month"] = monthly_precip["year_month_dt"].dt.month

In [25]:
fig = px.line(
    monthly_precip.sort_values("year_month_dt"),
    x="year_month_dt",
    y="pct_rain_hours",
    line_shape="spline",
    markers=False,
)

fig.update_traces(
    line=dict(color="#1f6feb", width=1.6),
    hovertemplate="Month: %{x|%Y-%m}<br>Rain hours: %{y:.1f}%<extra></extra>",
)

fig.update_xaxes(title_text="Year")
fig.update_yaxes(title_text="% of hours with rainfall")

apply_layout(
    fig,
    "Figure 1. Monthly percentage of rainfall hours throughout the period",
)

render_fig(fig)

**Interpretation:** The monthly percentage of rainy hours exhibits a clear seasonal pattern, with recurring peaks that repeat consistently from year to year, reflecting the typical summer rainfall regime of the region. This seasonality represents exactly the type of temporal variation that a static model cannot adapt to, providing the underlying motivation for comparing the three model update strategies.

<a id="sec-results-consolidation"></a>
## 4.2 Results Consolidation

After executing the three experimental scenarios, the results are consolidated at two complementary levels: an overall summary for each scenario and an annual summary by scenario. This organization enables the comparison of not only the average predictive performance but also the temporal stability of each model update strategy throughout the evaluation period.

In [26]:
summary = (
    results.groupby("scenario", as_index=False)
           .agg(tp=("tp", "sum"), tn=("tn", "sum"), fp=("fp", "sum"), fn=("fn", "sum"), windows=("window_idx", "count"))
)
summary = pd.concat([summary, compute_metrics_from_confusion(summary["tp"], summary["tn"], summary["fp"], summary["fn"])], axis=1)

yearly_results = (
    results.groupby(["scenario", "year"], as_index=False)
           .agg(tp=("tp", "sum"), tn=("tn", "sum"), fp=("fp", "sum"), fn=("fn", "sum"), windows=("window_idx", "count"))
)
yearly_metrics = compute_metrics_from_confusion(yearly_results["tp"], yearly_results["tn"], yearly_results["fp"], yearly_results["fn"])
yearly_results = pd.concat([yearly_results, yearly_metrics], axis=1)

<a id="sec-aggregated-results"></a>
## 4.3 Interpreting the Aggregated Results

The overall summary provides an initial view of predictive performance, but it should not be interpreted in isolation. In long-term time series experiments, yearly behavior is as important as the final average, because a scenario may appear competitive in the aggregate while still exhibiting poor stability over time.

For this reason, the primary interpretation in this study relies on the annual evolution of the F1-score and on the paired statistical comparisons between the evaluated scenarios.

In [19]:
display(summary.sort_values("f1", ascending=False))

,scenario,tp,tn,fp,fn,windows,accuracy,precision,recall,f1
0,Incremental,5204,111390,5594,6143,350,0.908541,0.481941,0.458623,0.469993
1,Periodic Retraining,4080,111011,5973,7267,350,0.896829,0.405849,0.359566,0.381308
2,Static Training,6624,95128,21856,4723,350,0.792887,0.232584,0.583767,0.332639


**Table 1.** Overall performance summary by scenario (accuracy, precision, recall, and F1-score aggregated across all evaluation windows).

**Interpretation:** Considering the F1-score computed over the aggregated predictions from all evaluation windows, the **Incremental Learning** scenario achieves the highest value among the three strategies. The **Static Training** scenario is notable for combining relatively high recall (0.584) with low precision (0.233), suggesting an increasing tendency to overpredict rainfall as the data distribution gradually diverges from that observed during the initial training period. However, this aggregated summary does not capture how model performance evolves over time; therefore, the following section examines the annual performance of each strategy.

In [30]:
fig = go.Figure()

for scenario, group in yearly_results.sort_values("year").groupby("scenario"):
    fig.add_trace(
        go.Scatter(
            x=group["year"],
            y=group["f1"],
            mode="lines+markers",
            name=scenario,
            line=dict(
                shape="spline",
                smoothing=0.4,
                width=2.5,
                color=SCENARIO_COLORS.get(scenario),
            ),
            marker=dict(
                size=7,
                color=SCENARIO_COLORS.get(scenario),
            ),
            hovertemplate="Year: %{x}<br>F1-score: %{y:.3f}<extra>"
            + scenario
            + "</extra>",
        )
    )

fig.update_xaxes(title_text="Year", dtick=1)
fig.update_yaxes(title_text="F1-score")

apply_layout(
    fig,
    "Figure 2. Annual F1-score evolution by scenario",
)

render_fig(fig)

**Interpretation:** The annual results confirm the pattern observed in the aggregated summary. The **Incremental Learning** scenario consistently achieves the highest F1-score in every year of the evaluation period, outperforming both the **Periodic Retraining** and **Static Training** strategies. The **Periodic Retraining** scenario systematically occupies an intermediate position, mitigating part of the performance degradation observed in the **Static Training** approach, yet remaining below the performance achieved through **Incremental Learning**. This consistency throughout the entire study period indicates that the superiority of the incremental strategy is not driven by a small number of favorable years, but rather reflects a stable and recurring pattern over time.

In [31]:
order = ["Static Training", "Periodic Retraining", "Incremental Learning"]

fig = go.Figure()

for scenario in order:
    values = yearly_results.loc[yearly_results["scenario"] == scenario, "f1"]

    fig.add_trace(
        go.Box(
            y=values,
            name=scenario,
            marker_color=SCENARIO_COLORS.get(scenario),
            boxmean=False,
            boxpoints="all",
            jitter=0.4,
            pointpos=0,
            hovertemplate="F1-score: %{y:.3f}<extra>"
            + scenario
            + "</extra>",
        )
    )

fig.update_yaxes(title_text="F1-score")

apply_layout(
    fig,
    "Figure 3. Distribution of annual F1-score by scenario",
    width=800,
)

fig.update_layout(showlegend=False)

render_fig(fig)

**Interpretation:** The boxplot summarizes both the overall performance level and its variability across the years. The **Incremental Learning** scenario presents the highest median F1-score, indicating superior typical performance. The **Static Training** scenario concentrates its values at a lower performance level, reflecting the degradation observed throughout the temporal series. The **Periodic Retraining** strategy occupies an intermediate position, with variability comparable to that of the incremental approach. This distribution-based analysis complements the temporal comparison and motivates the paired statistical test presented in the following section.

In [32]:
diff_pivot = yearly_results.pivot(
    index="year",
    columns="scenario",
    values="f1",
).sort_index()

diff_series = (
    diff_pivot["Incremental Learning"] - diff_pivot["Periodic Retraining"]
).dropna()

bar_colors = [
    "#2ca02c" if v >= 0 else "#d62728"
    for v in diff_series.values
]

fig = go.Figure(
    go.Bar(
        x=diff_series.index.astype(str),
        y=diff_series.values,
        marker_color=bar_colors,
        text=[f"{v:.2f}" for v in diff_series.values],
        textposition="outside",
        hovertemplate="Year: %{x}<br>Δ F1: %{y:.3f}<extra></extra>",
    )
)

fig.add_hline(y=0, line_color="black", line_width=1)

fig.update_xaxes(title_text="Year")
fig.update_yaxes(title_text="Δ F1-score")

apply_layout(
    fig,
    "Figure 4. Annual F1-score difference (Incremental Learning − Periodic Retraining)",
)

render_fig(fig)

**Interpretation:** The annual F1-score difference between **Incremental Learning** and **Periodic Retraining** is positive in every evaluated year, indicating that the advantage of incremental learning does not depend on a small number of atypical years, but instead appears consistently throughout the entire time series.

<a id="sec-statistical-comparison"></a>
## 4.4 Statistical Comparison Between Scenarios

To complement the visual analysis, a paired Wilcoxon signed-rank test was applied to the annual F1-score values. This non-parametric test is appropriate because the comparisons are performed between paired yearly observations without assuming that the differences follow a normal distribution.

The comparisons were conducted between all three pairs of scenarios, applying Bonferroni correction to control the error rate caused by multiple statistical tests. In addition to the adjusted p-value, the effect size **r** was also calculated, providing additional insight into the practical relevance of the observed differences.

In [36]:
def interpret_effect_size(r: float) -> str:
    """Classifies the magnitude of effect size r into qualitative categories.

    Args:
        r: Effect size calculated from the z-statistic of the Wilcoxon test.

    Returns:
        str: One of the categories "very small", "small", "moderate",
        "large", or "undefined" (when r is NaN).
    """
    if pd.isna(r):
        return "undefined"
    if abs(r) < 0.10:
        return "very small"
    if abs(r) < 0.30:
        return "small"
    if abs(r) < 0.50:
        return "moderate"
    return "large"


def wilcoxon_with_effect_size(
    yearly_results: pd.DataFrame,
    group_a: str,
    group_b: str,
    alpha: float = 0.05,
    n_comparisons: int = 1,
) -> Dict:
    """Compares two scenarios using a paired Wilcoxon test on annual F1-score.

    Args:
        yearly_results: DataFrame containing annual F1-score values by
            scenario, with at least the columns "year", "scenario", and "f1".
        group_a: Name of the first scenario to compare.
        group_b: Name of the second scenario to compare.
        alpha: Significance level before Bonferroni correction.
        n_comparisons: Total number of comparisons performed, used to adjust
            alpha through Bonferroni correction.

    Returns:
        Dict: Test statistic, raw and adjusted p-values, adjusted alpha,
        decision regarding H0 rejection, effect size r, and qualitative
        interpretation.

    Notes:
        Effect size r is derived from the approximate z-statistic of the
        Wilcoxon test, divided by the square root of the number of non-zero
        pairs, following the standard convention for this test.
    """
    pivot = (
        yearly_results
        .pivot(index="year", columns="scenario", values="f1")
        .dropna(subset=[group_a, group_b])
    )

    x = pivot[group_a].to_numpy()
    y = pivot[group_b].to_numpy()

    test = wilcoxon(
        x,
        y,
        zero_method="wilcox",
        alternative="two-sided",
        method="approx",
    )

    diff = x - y
    diff_nz = diff[diff != 0]
    n = len(diff_nz)

    if n == 0:
        r = np.nan
    else:
        ranks = pd.Series(np.abs(diff_nz)).rank(method="average").to_numpy()
        w_plus = np.sum(ranks[diff_nz > 0])

        mu_w = n * (n + 1) / 4
        sigma_w = np.sqrt(n * (n + 1) * (2 * n + 1) / 24)

        z = (w_plus - mu_w) / sigma_w if sigma_w > 0 else np.nan
        r = z / np.sqrt(n) if sigma_w > 0 else np.nan

    alpha_bonf = alpha / n_comparisons

    return {
        "comparison": f"{group_a} vs {group_b}",
        "n_pairs": len(pivot),
        "wilcoxon_stat": float(test.statistic),
        "pvalue": float(test.pvalue),
        "pvalue_bonferroni": min(float(test.pvalue) * n_comparisons, 1.0),
        "alpha_bonferroni": alpha_bonf,
        "reject_h0": float(test.pvalue) < alpha_bonf,
        "r": float(r) if pd.notna(r) else np.nan,
        "effect_magnitude": interpret_effect_size(r),
    }

<a id="sec-statistical-interpretation"></a>
## 4.5 Statistical Interpretation

The statistical interpretation should consider two complementary aspects. The first is statistical significance, represented by the adjusted p-value. The second is practical relevance, represented by the effect size.

This combination prevents superficial conclusions such as “the difference was statistically significant, therefore it is better.” In real-world applications, small differences may be statistically detectable while still providing limited operational benefit.

In [37]:
paired_f1 = yearly_results.pivot(
    index="year",
    columns="scenario",
    values="f1",
).sort_index()

comparisons = [
    ("Incremental Learning", "Static Training"),
    ("Incremental Learning", "Periodic Retraining"),
    ("Periodic Retraining", "Static Training"),
]

stat_results = pd.DataFrame([
    wilcoxon_with_effect_size(
        yearly_results,
        a,
        b,
        alpha=0.05,
        n_comparisons=len(comparisons),
    )
    for a, b in comparisons
])

display(stat_results)

,comparison,n_pairs,wilcoxon_stat,pvalue,pvalue_bonferroni,alpha_bonferroni,reject_h0,r,effect_magnitude
0,Incremental Learning vs Static Training,14,0.0,0.000982,0.002945,0.016667,True,0.880830,large
1,Incremental Learning vs Periodic Retraining,14,0.0,0.000982,0.002945,0.016667,True,0.880830,large
2,Periodic Retraining vs Static Training,14,8.0,0.005213,0.015640,0.016667,True,0.746609,large


**Table 2.** Paired Wilcoxon comparisons of annual F1-score values, with Bonferroni correction and effect size **r**.

**Interpretation:** In the paired Wilcoxon test, rejecting the null hypothesis (H0) indicates statistical evidence that the median of the annual F1-score differences between the scenarios is different from zero. All comparisons showed statistically significant differences with large effect sizes, confirming the performance hierarchy observed in the previous analyses (**Incremental Learning** > **Periodic Retraining** > **Static Training**).

<a id="sec-discussion"></a>
# 5. Discussion

The results indicate that the incremental learning strategy tends to maintain more consistent performance over time. This suggests that the temporal dynamics of the dataset are sufficiently relevant to penalize static approaches.

The static training scenario serves as an important reference point: it illustrates the consequences of using a model that does not adapt to changes in the underlying data distribution. The periodic retraining approach provides partial adaptation, but without the same continuous adjustment capability observed in the incremental learning scenario.

From a technical perspective, these findings reinforce a central idea of this study: **in real-world temporal data, the model update strategy has a practical impact on the final predictive performance**.

<a id="sec-limitations"></a>
## 5.1 Experiment Limitations

This study was designed to provide a clear and interpretable evaluation; however, some limitations should be explicitly acknowledged:

- the experiment considers only a single model family;
- the comparison depends on the specific temporal window configuration adopted;
- the annual analysis reduces the temporal granularity available for statistical comparisons;
- the dataset was kept imbalanced, which better reflects real-world conditions but increases the difficulty of the prediction task.

These limitations do not invalidate the experiment. Instead, they provide important context for interpreting the results and clarify the scope of the conclusions drawn from this study.

<a id="sec-main-results"></a>
## 5.2 Main Results

- The **Incremental Learning** scenario achieved the highest aggregated F1-score (0.470) among the three evaluated strategies.
- The **Static Training** scenario achieved the lowest aggregated F1-score (0.333) and showed evidence of performance degradation over time.
- **Periodic Retraining** improved performance compared to static training (F1 = 0.381), but remained below the incremental learning scenario.
- The Wilcoxon test results further support the superiority of the **Incremental Learning** strategy and confirm the intermediate performance of **Periodic Retraining** compared to **Static Training**.

<a id="sec-conclusion"></a>
# 6. Conclusion

This notebook compared three model update strategies for a binary hourly rainfall prediction task using a real-world temporal dataset. The analysis combined temporal exploration of the dataset, window-based evaluation, annual performance consolidation, and paired statistical testing.

Overall, the results show that **Incremental Learning** achieved the best average performance and the highest temporal consistency in F1-score among the evaluated scenarios. **Static Training** demonstrated lower adaptability over time, while **Periodic Retraining** occupied an intermediate position between the two approaches.

Rather than simply identifying a winning strategy, the main contribution of this study is to demonstrate that, in real-world temporal problems, **the model update policy is a fundamental component of the solution**.